# Experiment 1: scaffold-efficiency A/B on the real gemma-4-31b-it-qat-w4a16-ct

Four arms (control; edit-early prompt; thinking off; both) on the SAME 30 dev tasks, one vLLM boot, temperature 0.2, 5-minute cap (the submission's settings). Hypotheses, metrics and the decision rule are in docs/STRATEGY.md ('Experiment 1'). Motivation: in 104 prior attempts 63 hit the time cap (46 with NO patch), 96% of wall time was model latency, and the median first real edit came at tool call 10.

In [ ]:
import glob, importlib, os, shutil, subprocess, sys
from pathlib import Path
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try: os.unlink(pth)
        except OSError: pass
tmp = Path('/tmp/wheelhouse'); tmp.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower(): continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    t = tmp / name
    if not t.exists(): os.symlink(w, t)
wheels = sorted(str(w) for w in tmp.glob('*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
print('installed', len(wheels), 'wheels')

## Arms and dev tasks

In [ ]:
import json, random
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working'); WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Locate the arms dataset by content (agent.yaml name prefix), not by an assumed mount path.
arm_src = sorted({p.parent for p in Path('/kaggle/input').rglob('agent.yaml') if 'swe_exp_' in p.read_text(encoding='utf-8')})
assert len(arm_src) == 4, f'expected 4 arms, found {len(arm_src)}: {arm_src}'
ARMS = {}
for p in arm_src:
    dst = WORKING_DIR / 'arms' / p.name
    if dst.exists(): shutil.rmtree(dst)
    shutil.copytree(p, dst)
    for must in ('agent.yaml', 'prompts/system.md', 'prompts/analyzer.md', 'configs/sampling.yaml', 'sub_agents/code_analyzer.yaml', 'eval_config.yaml'):
        assert (dst / must).is_file(), f'arm {p.name} missing {must}'
    ARMS[p.name] = dst
print('arms:', list(ARMS))
for n, d in ARMS.items():
    print(n, '| sampling:', (d / 'configs/sampling.yaml').read_text().replace(chr(10), ' ')[-110:])

DEV_IDS = ["fastapi_14258", "fastapi_14372", "fastapi_14851", "fastapi_14873", "fastapi_14986", "fastapi_15280", "fastapi_15588", "fastapi_15589", "fastapi_15661", "fastapi_15800", "requests_7309", "requests_7315", "requests_7427", "rich_2943", "rich_3006", "rich_3043", "rich_3052", "rich_3063", "rich_3067", "rich_3180", "rich_3470", "rich_3506", "rich_3675", "rich_3772", "rich_3882", "rich_3894", "rich_3934", "rich_3944", "rich_4006", "rich_4075"]
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
by_id = {t.instance_id: t for t in load_tasks(TASKS_PATH)}
missing = [i for i in DEV_IDS if i not in by_id]
assert not missing, f'dev ids missing from dataset: {missing}'
print(f'{len(DEV_IDS)} dev tasks:', {r: sum(1 for i in DEV_IDS if i.startswith(r)) for r in ('fastapi', 'requests', 'rich')})

## vLLM (real 31B)

In [ ]:
import litellm, torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model
litellm.drop_params = True
TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
first_arm = ARMS[sorted(ARMS)[0]]
declared_model = validate_single_declared_model(first_arm)
adapters = discover_adapters(str(first_arm), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
vllm_cfg = VllmConfig(model=str(MODEL_PATH), port=8000, host='127.0.0.1', tool_call_parser='gemma4', reasoning_parser='gemma4',
    max_model_len=32768, dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90, enable_auto_tool_choice=True, enable_lora=True, max_loras=8, max_lora_rank=128,
    tensor_parallel_size=tp_size, startup_timeout=60 * 20)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')
models = server_instance.create_model_registry(aliases=[declared_model, TARGET_MODEL_NAME], model_prefix='openai/', api_key='EMPTY')

## Run all arms (same tasks, shuffled arm order)

In [ ]:
import asyncio, concurrent.futures, time, traceback, yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

def run_sync(coro_fn):
    try: loop = asyncio.get_running_loop()
    except RuntimeError: loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(coro_fn())).result()
    return asyncio.run(coro_fn())

def budget_of(arm_dir):
    cfg = yaml.safe_load((arm_dir / 'eval_config.yaml').read_text(encoding='utf-8')).get('evaluation', {})
    out = {}
    for k, cast in (('timeout_seconds', int), ('max_tool_calls', int), ('max_time_minutes', float)):
        if k in cfg: out[k] = cast(cfg[k])
    t = cfg.get('max_turns', cfg.get('max_llm_calls'))
    if t is not None: out['max_turns'] = int(t)
    return out

limits, gen_constraints = build_submission_limits()
EXP_DIR = WORKING_DIR / 'exp1'; EXP_DIR.mkdir(exist_ok=True)
order = sorted(ARMS); random.Random(11).shuffle(order)   # fixed shuffle: arm order is not systematically confounded with time
print('arm order this session:', order)
summary = {}
for arm in order:
    results_dir = EXP_DIR / arm
    cfg = EvalConfig(tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots', results_dir=results_dir,
        submission_dir=ARMS[arm], models=models, sandbox='subprocess', task_ids=DEV_IDS, concurrency=4, display_mode='quiet',
        limits=limits, generation_constraints=gen_constraints, adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5),
        graph_dir=str(DATA_DIR / 'graphs'), embeddings_dir=str(DATA_DIR / 'embeddings'), wheels_dir=DATA_DIR / 'wheels',
        verbose=False, **budget_of(ARMS[arm]))
    print(f'\n===== arm {arm} | budget {budget_of(ARMS[arm])} =====')
    t0 = time.perf_counter()
    try:
        res = run_sync(Evaluator(cfg).run)
    except Exception:
        traceback.print_exc(); summary[arm] = {'error': 'run raised'}; continue
    dur = time.perf_counter() - t0
    rows = [json.loads(l) for l in open(results_dir / 'task_results.jsonl', encoding='utf-8')]
    n = len(rows)
    summary[arm] = {
        'n': n, 'resolved': sum(r['resolved'] for r in rows),
        'patch_written': sum(1 for r in rows if r['agent_patch_size'] > 0),
        'time_cap_hit': sum(1 for r in rows if 'session timeout' in (r['error'] or '')),
        'mean_task_s': round(sum(r['duration_seconds'] for r in rows) / n, 1),
        'wall_s': round(dur, 1),
        'resolved_ids': sorted(r['instance_id'] for r in rows if r['resolved']),
    }
    print(json.dumps({k: v for k, v in summary[arm].items() if k != 'resolved_ids'}))
    (EXP_DIR / 'summary_all.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')   # incremental: survives a dead session

print('\n===== SUMMARY (same %d dev tasks per arm) =====' % len(DEV_IDS))
print(f"{'arm':16s} {'resolved':>8s} {'patch':>6s} {'capHit':>7s} {'mean_s':>7s}")
for arm in sorted(summary):
    s = summary[arm]
    if 'error' in s: print(arm, s); continue
    print(f"{arm:16s} {s['resolved']:>8d} {s['patch_written']:>6d} {s['time_cap_hit']:>7d} {s['mean_task_s']:>7.1f}")